# 01 — Data Exploration & Predictor-Dataset Pipeline (Corrected)

Goal: NOT blood-cell classification. This dataset provides realistic
feature values used to construct quantum circuits, so we can learn how
circuit/data/hardware characteristics affect noise (TVD between ideal
and noisy execution).

**Methodology (finalized, per Shreya's review):**
1. Exact ideal distribution (Statevector) vs. FakeTorino noisy
   simulation -- no Monte Carlo noise on the ideal side.
2. Fixed transpiler seed + deliberate, fully-crossed physical layout
   variation (layout is independent of data row and circuit config,
   not confounded with either).
3. Rich hardware-aware feature extraction (structural, transpiled,
   CZ calibration, T1/T2) -- kept in full, not reduced to just
   depth + CX count.
4. Features organized into three nested tiers (baseline / structural /
   hardware-aware) so we can directly test whether hardware-aware
   information improves noise prediction beyond circuit complexity --
   this is the paper's central experimental question.


In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Load and inspect the dataset

In [5]:
DATA_PATH = "../data/raw/blood_cell_anomaly_detection.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (5880, 36)


,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,granularity_score,lobularity_score,membrane_smoothness,cell_area_px,perimeter_px,mean_r,mean_g,mean_b,stain_intensity,patient_age_group,patient_sex,wbc_count_per_ul,rbc_count_millions_per_ul,hemoglobin_g_dl,hematocrit_pct,platelet_count_per_ul,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,4.11,6.6,0.800,445,90,215,141,160,0.555,Elderly,F,6352,4.44,11.7,43.4,257383,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,2.50,6.3,0.737,383,61,189,136,169,0.692,Elderly,M,7709,4.90,13.9,42.2,302274,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,3.01,3.2,0.790,418,52,193,150,203,0.593,Adult,M,7451,5.72,16.1,39.2,229996,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,0.43,1.0,0.937,217,39,227,138,118,0.509,Adult,F,9196,3.42,14.6,54.1,130720,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,0.51,1.0,0.925,147,50,239,127,113,0.607,Elderly,M,5898,5.36,14.6,36.7,228652,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5880 entries, 0 to 5879
Data columns (total 36 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   cell_id                                  5880 non-null   str    
 1   cell_type                                5880 non-null   str    
 2   anomaly_label                            5880 non-null   int64  
 3   disease_category                         5880 non-null   str    
 4   cell_diameter_um                         5880 non-null   float64
 5   nucleus_area_pct                         5880 non-null   float64
 6   chromatin_density                        5880 non-null   float64
 7   cytoplasm_ratio                          5880 non-null   float64
 8   circularity                              5880 non-null   float64
 9   eccentricity                             5880 non-null   float64
 10  granularity_score                        5880 non-null   fl

In [7]:
print(df.columns.tolist())


['cell_id', 'cell_type', 'anomaly_label', 'disease_category', 'cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score', 'lobularity_score', 'membrane_smoothness', 'cell_area_px', 'perimeter_px', 'mean_r', 'mean_g', 'mean_b', 'stain_intensity', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl', 'dataset_source', 'staining_protocol', 'microscope_model', 'magnification_x', 'image_resolution_px', 'cytodiffusion_anomaly_score', 'cytodiffusion_classification_confidence', 'labeller_confidence_score']


In [8]:
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0])

print("\nDuplicate rows:", df.duplicated().sum())


Columns with missing values:
Series([], dtype: int64)

Duplicate rows: 0


## Feature selection

Selected 5 morphology features (Group A from our discussion -- least
redundant with each other, directly describe individual cell structure,
no label leakage, no ID/metadata columns).


In [9]:
selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

feature_data = df[selected_columns].copy()

print(feature_data.describe().round(3).to_string())
print("\nMissing values:")
print(feature_data.isna().sum().to_string())
print("\nCorrelation:")
print(feature_data.corr().round(3).to_string())


       cell_diameter_um  chromatin_density  cytoplasm_ratio  circularity  eccentricity
count          5880.000           5880.000         5880.000     5880.000      5880.000
mean             10.176              0.391            0.564        0.769         0.365
std               3.642              0.310            0.335        0.158         0.202
min               1.000              0.000            0.050        0.100         0.000
25%               7.630              0.000            0.276        0.711         0.215
50%              10.055              0.480            0.452        0.805         0.331
75%              12.860              0.635            1.000        0.879         0.456
max              21.180              1.000            1.000        1.000         0.990

Missing values:
cell_diameter_um     0
chromatin_density    0
cytoplasm_ratio      0
circularity          0
eccentricity         0

Correlation:
                   cell_diameter_um  chromatin_density  cytoplasm_ratio

**Note on redundancy:** the `angle_i` features (created below) are a
deterministic linear rescaling of `raw_feature_i` -- they carry
identical information, just on a different scale. This is kept
intentionally (harmless for tree-based models like Random Forest /
XGBoost) but is worth knowing when interpreting feature importance --
importance may be split across both representations of the same
underlying signal.

## Scale features to [0, 2π] for angle encoding

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
scaled_features = scaler.fit_transform(feature_data.values)

print("Scaled shape:", scaled_features.shape)
print("Example scaled row:", scaled_features[0])


Scaled shape: (5880, 5)
Example scaled row: [4.415043   3.40548644 1.6600837  3.23234977 3.35737882]


## Build one example quantum kernel circuit

In [11]:
from qiskit.circuit.library import zz_feature_map

n_features = len(selected_columns)

feature_map = zz_feature_map(
    feature_dimension=n_features,
    reps=1,
    entanglement="linear",
)

print("Number of qubits:", feature_map.num_qubits)
print("Parameters:", feature_map.parameters)


Number of qubits: 5
Parameters: ParameterView([ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2]), ParameterVectorElement(x[3]), ParameterVectorElement(x[4])])


In [12]:
one_data_point = scaled_features[0]

bound_circuit = feature_map.assign_parameters(one_data_point)
print(bound_circuit.decompose())


   ┌────────────┐┌───────────────┐                                 »
0: ┤ U(π/2,0,π) ├┤ U(0,0,8.8301) ├──■──────────────────────■───────»
   ├────────────┤└┬──────────────┤┌─┴─┐┌────────────────┐┌─┴─┐     »
1: ┤ U(π/2,0,π) ├─┤ U(0,0,6.811) ├┤ X ├┤ U(0,0,0.67211) ├┤ X ├──■──»
   ├────────────┤┌┴──────────────┤└───┘└────────────────┘└───┘┌─┴─┐»
2: ┤ U(π/2,0,π) ├┤ U(0,0,3.3202) ├────────────────────────────┤ X ├»
   ├────────────┤├───────────────┤                            └───┘»
3: ┤ U(π/2,0,π) ├┤ U(0,0,6.4647) ├─────────────────────────────────»
   ├────────────┤├───────────────┤                                 »
4: ┤ U(π/2,0,π) ├┤ U(0,0,6.7148) ├─────────────────────────────────»
   └────────────┘└───────────────┘                                 »
«                                                             »
«0: ──────────────────────────────────────────────────────────»
«                                                             »
«1: ─────────────────────■───────────────────────

## Simulators and noise model

Using `FakeTorino` -- a real, frozen calibration snapshot of an actual
IBM device (T1, T2, gate errors). No live IBM account or network access
required.

In [13]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit.quantum_info import Statevector

fake_backend = FakeTorino()
print("Backend:", fake_backend.name)
print("Number of qubits:", fake_backend.num_qubits)

ideal_simulator = AerSimulator()
noise_model = NoiseModel.from_backend(fake_backend)
noisy_simulator = AerSimulator(noise_model=noise_model)


Backend: fake_torino
Number of qubits: 133


## TVD helper functions

`calculate_tvd_from_probs` operates directly on probability dicts (used
for the exact ideal distribution, which has no "counts"). `calculate_tvd`
is kept for convenience when comparing two counts dicts directly (e.g.
for quick manual checks).

In [14]:
def counts_to_probabilities(counts):
    total_shots = sum(counts.values())
    return {state: count / total_shots for state, count in counts.items()}


def calculate_tvd_from_probs(prob1, prob2):
    all_states = set(prob1) | set(prob2)
    p1 = np.array([prob1.get(s, 0) for s in all_states])
    p2 = np.array([prob2.get(s, 0) for s in all_states])
    return 0.5 * np.sum(np.abs(p1 - p2))


def calculate_tvd(counts1, counts2):
    return calculate_tvd_from_probs(
        counts_to_probabilities(counts1),
        counts_to_probabilities(counts2),
    )


### FIX 1 demonstrated: exact ideal distribution vs. shot-sampled

Previously, the "ideal" reference was estimated via 1024 shots -- itself
noisy. Here it's computed exactly via `Statevector`, with zero shots and
zero randomness.

In [15]:
measured_circuit = bound_circuit.copy()
measured_circuit.measure_all()

# EXACT ideal distribution -- no shots needed, no shot noise
ideal_sv = Statevector.from_instruction(bound_circuit)  # unmeasured circuit
prob_ideal_exact = ideal_sv.probabilities_dict()

print("Exact ideal distribution (first few states):")
for state, prob in list(prob_ideal_exact.items())[:5]:
    print(f"  {state}: {prob:.4f}")

# Noisy simulation (still shot-based -- this is correct, real hardware
# noise genuinely IS stochastic and should be sampled)
noisy_compiled = transpile(measured_circuit, noisy_simulator)
counts_noisy = noisy_simulator.run(noisy_compiled, shots=1024).result().get_counts()
prob_noisy = counts_to_probabilities(counts_noisy)

tvd = calculate_tvd_from_probs(prob_ideal_exact, prob_noisy)
print("\nTVD (exact ideal vs. noisy):", tvd)


Exact ideal distribution (first few states):
  00000: 0.0312
  00001: 0.0312
  00010: 0.0312
  00011: 0.0312
  00100: 0.0312

TVD (exact ideal vs. noisy): 0.07421875


## Circuit structural feature extraction (abstract circuit)

In [16]:
from collections import Counter


def extract_circuit_features(circuit):
    """Structural / topology features from a concrete QuantumCircuit."""
    qc = circuit.decompose()
    ops = qc.count_ops()

    n_qubits = qc.num_qubits
    depth = qc.depth()
    total_gate_count = sum(ops.values())

    two_qubit_gate_names = {"cx", "cz", "ecr", "swap"}
    two_qubit_gate_count = sum(
        count for gate, count in ops.items() if gate in two_qubit_gate_names
    )
    single_qubit_gate_count = total_gate_count - two_qubit_gate_count
    cx_count = ops.get("cx", 0)

    two_qubit_gate_ratio = (
        two_qubit_gate_count / total_gate_count if total_gate_count else 0.0
    )
    average_gates_per_qubit = total_gate_count / n_qubits if n_qubits else 0.0

    qubit_gate_counts = Counter()
    pair_counts = Counter()

    for instruction in qc.data:
        qubits = instruction.qubits
        if len(qubits) == 1:
            q = qc.find_bit(qubits[0]).index
            qubit_gate_counts[q] += 1
        elif len(qubits) == 2:
            q1 = qc.find_bit(qubits[0]).index
            q2 = qc.find_bit(qubits[1]).index
            qubit_gate_counts[q1] += 1
            qubit_gate_counts[q2] += 1
            pair = tuple(sorted((q1, q2)))
            pair_counts[pair] += 1

    unique_2q_pairs = len(pair_counts)
    repeated_2q_pairs = sum(count - 1 for count in pair_counts.values())

    neighbors = {q: set() for q in range(n_qubits)}
    for q1, q2 in pair_counts:
        neighbors[q1].add(q2)
        neighbors[q2].add(q1)

    degrees = [len(neighbors[q]) for q in range(n_qubits)]
    average_qubit_degree = np.mean(degrees) if degrees else 0.0
    max_qubit_degree = max(degrees) if degrees else 0
    max_gates_per_qubit = max(qubit_gate_counts.values()) if qubit_gate_counts else 0

    return {
        "n_qubits": n_qubits,
        "depth": depth,
        "total_gate_count": total_gate_count,
        "single_qubit_gate_count": single_qubit_gate_count,
        "two_qubit_gate_count": two_qubit_gate_count,
        "cx_count": cx_count,
        "two_qubit_gate_ratio": two_qubit_gate_ratio,
        "average_gates_per_qubit": average_gates_per_qubit,
        "max_gates_per_qubit": max_gates_per_qubit,
        "unique_2q_pairs": unique_2q_pairs,
        "average_qubit_degree": average_qubit_degree,
        "max_qubit_degree": max_qubit_degree,
        "repeated_2q_pairs": repeated_2q_pairs,
    }


## Transpiled-circuit feature extraction

In [17]:
def extract_transpiled_features(circuit, backend):
    qc = circuit.remove_final_measurements(inplace=False)
    ops = qc.count_ops()

    one_qubit_gates = {"sx", "x", "rz", "id", "u", "u1", "u2", "u3"}
    two_qubit_gates = {"cx", "cz", "ecr", "swap"}

    total_gates = sum(ops.values())
    one_qubit_count = sum(ops.get(gate, 0) for gate in one_qubit_gates)
    two_qubit_count = sum(ops.get(gate, 0) for gate in two_qubit_gates)

    return {
        "transpiled_depth": qc.depth(),
        "transpiled_total_gates": total_gates,
        "transpiled_1q_gates": one_qubit_count,
        "transpiled_2q_gates": two_qubit_count,
        "transpiled_cz_count": ops.get("cz", 0),
        "transpiled_cx_count": ops.get("cx", 0),
        "transpiled_sx_count": ops.get("sx", 0),
        "transpiled_rz_count": ops.get("rz", 0),
        "backend_qubits": backend.num_qubits,
    }


## Hardware calibration feature extraction (real CZ error / duration data)

In [18]:
def extract_hardware_noise_features(circuit, backend):
    """Hardware-aware features from a transpiled circuit's actual
    physical CZ couplers, using the backend's real calibration data."""
    cz_properties = backend.target["cz"]

    cz_errors = []
    cz_durations = []

    for instruction in circuit.data:
        if instruction.operation.name != "cz":
            continue
        q1 = circuit.find_bit(instruction.qubits[0]).index
        q2 = circuit.find_bit(instruction.qubits[1]).index
        pair = (q1, q2)
        prop = cz_properties.get(pair)
        if prop is None:
            pair = (q2, q1)
            prop = cz_properties.get(pair)
        if prop is not None and prop.error is not None:
            cz_errors.append(prop.error)
        if prop is not None and prop.duration is not None:
            cz_durations.append(prop.duration)

    features = {}
    if cz_errors:
        features["mean_cz_error"] = np.mean(cz_errors)
        features["std_cz_error"] = np.std(cz_errors)
        features["min_cz_error"] = np.min(cz_errors)
        features["max_cz_error"] = np.max(cz_errors)
        features["sum_cz_error"] = np.sum(cz_errors)
        survival = np.prod([1 - e for e in cz_errors])
        features["cumulative_cz_error"] = 1 - survival
    else:
        features.update({
            "mean_cz_error": 0.0, "std_cz_error": 0.0, "min_cz_error": 0.0,
            "max_cz_error": 0.0, "sum_cz_error": 0.0, "cumulative_cz_error": 0.0,
        })

    if cz_durations:
        features["mean_cz_duration"] = np.mean(cz_durations)
        features["max_cz_duration"] = np.max(cz_durations)
        features["total_cz_duration"] = np.sum(cz_durations)
    else:
        features.update({
            "mean_cz_duration": 0.0, "max_cz_duration": 0.0, "total_cz_duration": 0.0,
        })

    return features


## Consolidated `extract_all_features`

(Single final version -- earlier exploratory redefinitions removed.)

In [19]:
def extract_all_features(
    abstract_circuit,
    transpiled_circuit,
    backend,
    raw_data_row,
    data_row,
    reps,
    entanglement,
):
    """
    Extract all predictor features for one experiment.

    raw_data_row : original biological feature values.
    data_row     : scaled values used as quantum encoding angles.
    """
    features = {}

    # 1. Raw data features
    for i, value in enumerate(raw_data_row):
        features[f"raw_feature_{i+1}"] = float(value)

    # 2. Quantum encoding angles
    for i, value in enumerate(data_row):
        features[f"angle_{i+1}"] = float(value)

    # 3. Abstract circuit features
    abstract_features = extract_circuit_features(abstract_circuit)
    features.update({f"abstract_{k}": v for k, v in abstract_features.items()})

    # 4. Circuit configuration
    features["reps"] = reps
    features["entanglement"] = entanglement

    # 5. Transpiled circuit features
    transpiled_features = extract_transpiled_features(transpiled_circuit, backend)
    transpiled_features["backend_size"] = transpiled_features.pop("backend_qubits")
    features.update(transpiled_features)

    # 6. Hardware CZ error features
    hardware_features = extract_hardware_noise_features(transpiled_circuit, backend)
    features.update(hardware_features)

    # 7. Physical qubits used
    used_qubits = set()
    for instruction in transpiled_circuit.data:
        for qubit in instruction.qubits:
            q = transpiled_circuit.find_bit(qubit).index
            used_qubits.add(q)
    features["physical_qubit_count"] = len(used_qubits)
    features["physical_qubits_used"] = str(sorted(used_qubits))  # NEW: explicit record

    # 8. T1 / T2 features
    t1_values, t2_values = [], []
    for q in used_qubits:
        props = backend.qubit_properties(q)
        if props is None:
            continue
        if props.t1 is not None:
            t1_values.append(props.t1)
        if props.t2 is not None:
            t2_values.append(props.t2)

    if t1_values:
        features["mean_t1"] = float(np.mean(t1_values))
        features["min_t1"] = float(np.min(t1_values))
        features["max_t1"] = float(np.max(t1_values))
    else:
        features["mean_t1"] = np.nan
        features["min_t1"] = np.nan
        features["max_t1"] = np.nan

    if t2_values:
        features["mean_t2"] = float(np.mean(t2_values))
        features["min_t2"] = float(np.min(t2_values))
        features["max_t2"] = float(np.max(t2_values))
    else:
        features["mean_t2"] = np.nan
        features["min_t2"] = np.nan
        features["max_t2"] = np.nan

    return features


## Consolidated `run_complete_experiment` -- FIXED version

**Changes vs. the original:**
- Ideal distribution computed **once**, exactly, via `Statevector` --
  moved outside the repeat loop entirely (Fix 1).
- `seed_transpiler` fixed for reproducibility; optional `initial_layout`
  parameter added so different circuits can be deliberately mapped onto
  different physical qubits, giving real variation in the hardware
  calibration features rather than the transpiler always picking the
  same "best" patch (Fix 2).
- Only the noisy side is resampled `repeats` times -- correctly, since
  real hardware noise genuinely is stochastic and should be sampled,
  unlike the ideal reference.

In [20]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    shots=1024,
    repeats=10,
    backend=fake_backend,
    initial_layout=None,
    seed_transpiler=42,
):
    """
    Run one circuit configuration and return one complete
    predictor-dataset record.
    """

    # 1. Build abstract feature-map circuit, bind real data
    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )
    abstract_circuit = feature_map.assign_parameters(data_row)

    # 2. EXACT ideal distribution -- computed once, no shots, no randomness
    ideal_sv = Statevector.from_instruction(abstract_circuit)
    prob_ideal_exact = ideal_sv.probabilities_dict()

    # 3. Add measurements for the noisy/hardware-facing version
    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # 4. Canonical transpilation -- fixed seed, optional explicit layout
    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # 5. Extract all predictor features
    features = extract_all_features(
        abstract_circuit=abstract_circuit,
        transpiled_circuit=transpiled_circuit,
        backend=backend,
        raw_data_row=raw_data_row,
        data_row=data_row,
        reps=reps,
        entanglement=entanglement,
    )

    # 6. Noise model + noisy simulator (from same target backend)
    noise_model = NoiseModel.from_backend(backend)
    noisy_sim = AerSimulator(noise_model=noise_model)

    # 7. Repeated NOISY execution only -- ideal is fixed/exact (Fix 1)
    tvds = []
    for _ in range(repeats):
        noisy_result = noisy_sim.run(transpiled_circuit, shots=shots).result()
        counts_noisy = noisy_result.get_counts()
        prob_noisy = counts_to_probabilities(counts_noisy)
        tvd = calculate_tvd_from_probs(prob_ideal_exact, prob_noisy)
        tvds.append(tvd)

    # 8. Target statistics
    features["mean_tvd"] = float(np.mean(tvds))
    features["std_tvd"] = float(np.std(tvds, ddof=1))
    features["min_tvd"] = float(np.min(tvds))
    features["max_tvd"] = float(np.max(tvds))

    # 9. Experiment settings (now includes reproducibility info)
    features["shots"] = shots
    features["repeats"] = repeats
    features["seed_transpiler"] = seed_transpiler
    features["initial_layout"] = str(initial_layout)

    return features


## Test: run one complete experiment

In [21]:
raw_data_point = feature_data.iloc[0].to_numpy()
scaled_data_point = scaled_features[0]

complete_result = run_complete_experiment(
    raw_data_row=raw_data_point,
    data_row=scaled_data_point,
    reps=1,
    entanglement="full",
    shots=1024,
    repeats=10,
)

for key, value in complete_result.items():
    print(f"{key:30s}: {value}")


raw_feature_1                 : 15.18
raw_feature_2                 : 0.542
raw_feature_3                 : 0.301
raw_feature_4                 : 0.563
raw_feature_5                 : 0.529
angle_1                       : 4.415042995827875
angle_2                       : 3.4054864364913358
angle_3                       : 1.6600836969495538
angle_4                       : 3.232349774693498
angle_5                       : 3.3573788156545468
abstract_n_qubits             : 5
abstract_depth                : 23
abstract_total_gate_count     : 40
abstract_single_qubit_gate_count: 20
abstract_two_qubit_gate_count : 20
abstract_cx_count             : 20
abstract_two_qubit_gate_ratio : 0.5
abstract_average_gates_per_qubit: 8.0
abstract_max_gates_per_qubit  : 14
abstract_unique_2q_pairs      : 10
abstract_average_qubit_degree : 4.0
abstract_max_qubit_degree     : 4
abstract_repeated_2q_pairs    : 10
reps                          : 1
entanglement                  : full
transpiled_depth          

### Note on `initial_layout` diversity

By default (`initial_layout=None`), the transpiler will likely pick a
similar "best" patch of physical qubits every time for a small 5-qubit
circuit -- which can make `mean_cz_error` / `mean_t1` / `mean_t2` nearly
constant across your dataset. When building the full dataset below,
cycle through a handful of different explicit `initial_layout` values
(e.g., different regions of the 133-qubit chip) so these features
actually carry a learnable signal.

In [22]:
# Example: check what physical qubits got used by default
print("Physical qubits used (default layout):", complete_result["physical_qubits_used"])
# See the "Building the real predictor-training dataset" section below
# for the full, validated CANDIDATE_LAYOUTS list and the fully-crossed
# row x config x layout experimental design.


Physical qubits used (default layout): [27, 28, 29, 30, 36]


## Building the real predictor-training dataset

**Sampling strategy** (updated per Shreya's review):
- ~200 data rows, stratified by `cell_type`
- Circuit config grid: `reps ∈ {1,2,3}` × `entanglement ∈ {linear, full}`
  = 6 configs
- **Layout is fully crossed, not cycled by row index** -- every
  (row, config) pair is run under *every* candidate layout. This avoids
  confounding data identity with hardware placement, which matters if
  we later want to claim hardware-aware features improve prediction.
- Ideal distribution is exact (Fix 1) and doesn't depend on layout, so
  it only needs to be computed once per (row, config) pair, not
  recomputed per layout.
- `repeats=5` for the full sweep (not 10) to keep the total experiment
  count manageable -- a smaller validation subset can use more repeats
  later for tighter uncertainty estimates.

**Scale check:** 200 rows × 6 configs × 4 layouts = 4,800 canonical
circuits × 5 repeats = 24,000 noisy simulations. This is meaningfully
larger than a single-layout design -- **pilot on a small slice first**
(see next cell) before committing to the full run.

In [ ]:
# Before trusting the candidate layouts, check they're actually valid,
# reasonably-connected regions on FakeTorino's real coupling map --
# otherwise the transpiler may silently insert long SWAP chains to
# route between poorly-connected qubits, which would confound layout
# comparisons with routing-overhead artifacts rather than pure
# calibration differences.

coupling_map = fake_backend.coupling_map
print("Coupling map edges (first 20):", list(coupling_map.get_edges())[:20])

def layout_is_well_connected(layout, coupling_map, max_qubit_index):
    """Rough sanity check: are consecutive qubits in this layout
    directly connected on the real chip?"""
    if layout is None:
        return True
    if max(layout) >= max_qubit_index:
        return False
    edges = set(coupling_map.get_edges())
    connected_pairs = sum(
        1 for i in range(len(layout) - 1)
        if (layout[i], layout[i+1]) in edges or (layout[i+1], layout[i]) in edges
    )
    return connected_pairs >= (len(layout) - 1) * 0.5  # at least half directly connected

CANDIDATE_LAYOUTS = [
    None,                        # let transpiler choose (baseline)
    [0, 1, 2, 3, 4],
    [20, 21, 22, 23, 24]
]

for layout in CANDIDATE_LAYOUTS:
    ok = layout_is_well_connected(layout, coupling_map, fake_backend.num_qubits)
    print(f"Layout {layout}: {'OK' if ok else 'CHECK -- may route poorly, consider replacing'}")


Coupling map edges (first 20): [(0, 1), (0, 15), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (4, 3), (4, 5), (4, 16), (5, 4), (5, 6), (6, 5), (6, 7), (7, 6), (7, 8), (8, 7), (8, 9), (8, 17)]
Layout None: OK
Layout [0, 1, 2, 3, 4]: OK
Layout [20, 21, 22, 23, 24]: OK
Layout [50, 51, 52, 53, 54]: OK


In [27]:
coupling_edges = {
    tuple(sorted(edge))
    for edge in fake_backend.coupling_map.get_edges()
}

valid_layouts = []

for start in range(fake_backend.num_qubits):
    layout = list(range(start, start + 5))

    if max(layout) >= fake_backend.num_qubits:
        break

    connected = all(
        tuple(sorted((layout[i], layout[i + 1]))) in coupling_edges
        for i in range(4)
    )

    if connected:
        valid_layouts.append(layout)

print("Found connected 5-qubit chains:")
for layout in valid_layouts[:20]:
    print(layout)

Found connected 5-qubit chains:
[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[3, 4, 5, 6, 7]
[4, 5, 6, 7, 8]
[5, 6, 7, 8, 9]
[6, 7, 8, 9, 10]
[7, 8, 9, 10, 11]
[8, 9, 10, 11, 12]
[9, 10, 11, 12, 13]
[10, 11, 12, 13, 14]
[19, 20, 21, 22, 23]
[20, 21, 22, 23, 24]
[21, 22, 23, 24, 25]
[22, 23, 24, 25, 26]
[23, 24, 25, 26, 27]
[24, 25, 26, 27, 28]
[25, 26, 27, 28, 29]
[26, 27, 28, 29, 30]
[27, 28, 29, 30, 31]


In [28]:
CANDIDATE_LAYOUTS = [
    [0, 1, 2, 3, 4],
    [19, 20, 21, 22, 23],
    [38, 39, 40, 41, 42],
    [57, 58, 59, 60, 61],
]

for layout in CANDIDATE_LAYOUTS:
    print(layout, layout in valid_layouts)

[0, 1, 2, 3, 4] True
[19, 20, 21, 22, 23] True
[38, 39, 40, 41, 42] True
[57, 58, 59, 60, 61] True


In [30]:
import numpy as np
import time

# Get the same first pilot row
raw_test_row = pilot_sample_df[selected_columns].iloc[0].to_numpy()

# Scale it using the scaler you already fitted
scaled_test_row = scaler.transform(
    raw_test_row.reshape(1, -1)
)[0]

print("Raw:", raw_test_row)
print("Scaled:", scaled_test_row)
test_result_start = time.time()

test_result = run_complete_experiment(
    raw_data_row=raw_test_row,
    data_row=scaled_test_row,
    reps=1,
    entanglement="linear",
    shots=1024,
    repeats=1,
    initial_layout=[57, 58, 59, 60, 61],
    seed_transpiler=42,
)

test_elapsed = time.time() - test_result_start

print(f"Single experiment time: {test_elapsed:.2f} s")
print(f"Mean TVD: {test_result['mean_tvd']:.4f}")
print(f"Transpiled depth: {test_result['transpiled_depth']}")
print(f"Transpiled 2Q gates: {test_result['transpiled_2q_gates']}")
print(f"CZ count: {test_result['transpiled_cz_count']}")

Raw: [13.35   0.536  0.386  0.806  0.351]
Scaled: [3.84525959 3.36778732 2.22226343 4.92880981 2.22767479]
Single experiment time: 12.01 s
Mean TVD: 0.0605
Transpiled depth: 45
Transpiled 2Q gates: 8
CZ count: 8


c:\Users\jaspr\miniconda3\envs\qnoise\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\jaspr\miniconda3\envs\qnoise\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### Pilot run -- time a small slice before committing to the full sweep

Run on a small number of rows first (e.g. 5) with the full config ×
layout grid, and extrapolate the runtime to the full 200 rows before
committing.

In [31]:
import time

CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

df_with_features = (
    df.dropna(subset=selected_columns)
      .reset_index(drop=True)
)

PILOT_N_ROWS = 5

pilot_sample_df = (
    df_with_features
    .sample(n=PILOT_N_ROWS, random_state=42)
    .reset_index(drop=True)
)

print("Pilot sample shape:", pilot_sample_df.shape)
print("\nCell types:")
print(pilot_sample_df["cell_type"].value_counts())

pilot_raw = pilot_sample_df[selected_columns].to_numpy()
pilot_scaled = scaler.transform(pilot_raw)

# ------------------------------------------------------------
# Pilot experiment
# ------------------------------------------------------------

start = time.time()
pilot_results = []

total_experiments = (
    len(pilot_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

experiment_number = 0

for row_idx in range(len(pilot_raw)):

    for reps, entanglement in CONFIGS:

        for layout_idx, layout in enumerate(CANDIDATE_LAYOUTS):

            experiment_number += 1

            print(
                f"\n[{experiment_number}/{total_experiments}] "
                f"row={row_idx}, "
                f"reps={reps}, "
                f"entanglement={entanglement}, "
                f"layout={layout}"
            )

            exp_start = time.time()

            result = run_complete_experiment(
                raw_data_row=pilot_raw[row_idx],
                data_row=pilot_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            elapsed_exp = time.time() - exp_start

            result["data_row_index"] = row_idx
            result["layout_index"] = layout_idx

            pilot_results.append(result)

            print(
                f"Completed in {elapsed_exp:.1f}s | "
                f"mean_TVD={result['mean_tvd']:.4f}"
            )

# ------------------------------------------------------------
# Timing summary
# ------------------------------------------------------------

elapsed = time.time() - start

n_experiments_pilot = len(pilot_results)

n_experiments_full = (
    200
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

seconds_per_experiment = (
    elapsed / n_experiments_pilot
)

estimated_full_minutes = (
    seconds_per_experiment
    * n_experiments_full
    / 60
)

print("\n" + "=" * 60)
print("PILOT COMPLETE")
print("=" * 60)

print(
    f"Pilot experiments: {n_experiments_pilot}"
)

print(
    f"Total time: {elapsed / 60:.2f} minutes"
)

print(
    f"Average time/experiment: "
    f"{seconds_per_experiment:.2f} seconds"
)

print(
    f"Estimated full run: "
    f"{estimated_full_minutes:.1f} minutes"
)

Pilot sample shape: (5, 36)

Cell types:
cell_type
Neutrophil       2
Prolymphocyte    1
Lymphocyte       1
Platelet         1
Name: count, dtype: int64

[1/120] row=0, reps=1, entanglement=linear, layout=[0, 1, 2, 3, 4]
Completed in 36.9s | mean_TVD=0.0662

[2/120] row=0, reps=1, entanglement=linear, layout=[19, 20, 21, 22, 23]
Completed in 23.6s | mean_TVD=0.0658

[3/120] row=0, reps=1, entanglement=linear, layout=[38, 39, 40, 41, 42]
Completed in 26.9s | mean_TVD=0.0705

[4/120] row=0, reps=1, entanglement=linear, layout=[57, 58, 59, 60, 61]
Completed in 27.0s | mean_TVD=0.0684

[5/120] row=0, reps=2, entanglement=linear, layout=[0, 1, 2, 3, 4]
Completed in 73.5s | mean_TVD=0.0906

[6/120] row=0, reps=2, entanglement=linear, layout=[19, 20, 21, 22, 23]
Completed in 39.9s | mean_TVD=0.4133

[7/120] row=0, reps=2, entanglement=linear, layout=[38, 39, 40, 41, 42]
Completed in 26.2s | mean_TVD=0.3135

[8/120] row=0, reps=2, entanglement=linear, layout=[57, 58, 59, 60, 61]
Completed in 3

KeyboardInterrupt: 

In [25]:
for layout in CANDIDATE_LAYOUTS:
    print("\nLayout:", layout)

    if layout is None:
        print("Auto layout — skip connectivity check")
        continue

    edges = {
        tuple(sorted(edge))
        for edge in fake_backend.coupling_map.get_edges()
    }

    for i in range(len(layout) - 1):
        pair = tuple(sorted((layout[i], layout[i + 1])))

        print(
            f"{layout[i]} <-> {layout[i+1]}:",
            "CONNECTED" if pair in edges else "NOT CONNECTED"
        )


Layout: None
Auto layout — skip connectivity check

Layout: [0, 1, 2, 3, 4]
0 <-> 1: CONNECTED
1 <-> 2: CONNECTED
2 <-> 3: CONNECTED
3 <-> 4: CONNECTED

Layout: [20, 21, 22, 23, 24]
20 <-> 21: CONNECTED
21 <-> 22: CONNECTED
22 <-> 23: CONNECTED
23 <-> 24: CONNECTED

Layout: [50, 51, 52, 53, 54]
50 <-> 51: CONNECTED
51 <-> 52: CONNECTED
52 <-> 53: NOT CONNECTED
53 <-> 54: NOT CONNECTED


In [32]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

# Build ONCE
noise_model = NoiseModel.from_backend(fake_backend)

# Build ONCE
noisy_simulator = AerSimulator(
    noise_model=noise_model
)

print("Reusable noisy simulator ready.")

Reusable noisy simulator ready.


In [33]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    shots=1024,
    repeats=5,
    backend=fake_backend,
    noisy_sim=noisy_simulator,
    initial_layout=None,
    seed_transpiler=42,
):
    """
    Run one circuit configuration.

    Optimized version:
    - Exact Statevector ideal distribution
    - One canonical transpilation
    - Reuses a pre-built noisy Aer simulator
    - Executes all noisy repetitions as one Aer job
    """

    # ==========================================================
    # 1. Build abstract feature-map circuit
    # ==========================================================

    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(
        data_row
    )

    # ==========================================================
    # 2. EXACT ideal distribution
    # ==========================================================

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    prob_ideal_exact = ideal_sv.probabilities_dict()

    # ==========================================================
    # 3. Add measurements
    # ==========================================================

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # ==========================================================
    # 4. Canonical FakeTorino transpilation
    # ==========================================================

    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # ==========================================================
    # 5. Extract predictor features
    # ==========================================================

    features = extract_all_features(
        abstract_circuit=abstract_circuit,
        transpiled_circuit=transpiled_circuit,
        backend=backend,
        raw_data_row=raw_data_row,
        data_row=data_row,
        reps=reps,
        entanglement=entanglement,
    )

    # ==========================================================
    # 6. Run ALL noisy repetitions in ONE Aer job
    # ==========================================================

    circuits_to_run = [
        transpiled_circuit
        for _ in range(repeats)
    ]

    result = noisy_sim.run(
        circuits_to_run,
        shots=shots,
    ).result()

    # ==========================================================
    # 7. Calculate TVD for each repetition
    # ==========================================================

    tvds = []

    for i in range(repeats):

        counts_noisy = result.get_counts(i)

        prob_noisy = counts_to_probabilities(
            counts_noisy
        )

        tvd = calculate_tvd_from_probs(
            prob_ideal_exact,
            prob_noisy
        )

        tvds.append(tvd)

    # ==========================================================
    # 8. Target statistics
    # ==========================================================

    features["mean_tvd"] = float(
        np.mean(tvds)
    )

    features["std_tvd"] = float(
        np.std(tvds, ddof=1)
        if repeats > 1
        else 0.0
    )

    features["min_tvd"] = float(
        np.min(tvds)
    )

    features["max_tvd"] = float(
        np.max(tvds)
    )

    # ==========================================================
    # 9. Experiment metadata
    # ==========================================================

    features["shots"] = shots
    features["repeats"] = repeats
    features["seed_transpiler"] = seed_transpiler
    features["initial_layout"] = str(initial_layout)

    return features

In [34]:
import time

raw_test_row = pilot_sample_df[
    selected_columns
].iloc[0].to_numpy()

scaled_test_row = scaler.transform(
    raw_test_row.reshape(1, -1)
)[0]

start = time.time()

test_result = run_complete_experiment(
    raw_data_row=raw_test_row,
    data_row=scaled_test_row,
    reps=1,
    entanglement="linear",
    shots=1024,
    repeats=5,
    initial_layout=[57, 58, 59, 60, 61],
    seed_transpiler=42,
)

elapsed = time.time() - start

print(f"Time: {elapsed:.2f} seconds")
print(f"Mean TVD: {test_result['mean_tvd']:.4f}")
print(f"Std TVD: {test_result['std_tvd']:.4f}")
print(f"Depth: {test_result['transpiled_depth']}")
print(f"2Q gates: {test_result['transpiled_2q_gates']}")

Time: 3.56 seconds
Mean TVD: 0.0711
Std TVD: 0.0164
Depth: 45
2Q gates: 8


In [35]:
import time

start = time.time()

benchmark_results = []

row_idx = 0
reps = 1
entanglement = "linear"

for layout in CANDIDATE_LAYOUTS:

    exp_start = time.time()

    result = run_complete_experiment(
        raw_data_row=raw_test_row,
        data_row=scaled_test_row,
        reps=reps,
        entanglement=entanglement,
        shots=1024,
        repeats=5,
        initial_layout=layout,
        seed_transpiler=42,
    )

    elapsed = time.time() - exp_start

    benchmark_results.append(result)

    print(
        f"Layout {layout}: "
        f"{elapsed:.2f}s | "
        f"TVD={result['mean_tvd']:.4f}"
    )

print(
    f"\nTotal: {time.time() - start:.2f}s"
)

Layout [0, 1, 2, 3, 4]: 1.61s | TVD=0.0676
Layout [19, 20, 21, 22, 23]: 1.71s | TVD=0.0660
Layout [38, 39, 40, 41, 42]: 1.04s | TVD=0.0633
Layout [57, 58, 59, 60, 61]: 1.64s | TVD=0.0643

Total: 6.00s


In [36]:
for reps, entanglement in [
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]:
    start = time.time()

    result = run_complete_experiment(
        raw_data_row=raw_test_row,
        data_row=scaled_test_row,
        reps=reps,
        entanglement=entanglement,
        shots=1024,
        repeats=5,
        initial_layout=[57, 58, 59, 60, 61],
        seed_transpiler=42,
    )

    print(
        f"{reps}, {entanglement}: "
        f"{time.time()-start:.2f}s | "
        f"TVD={result['mean_tvd']:.4f}"
    )

2, linear: 1.71s | TVD=0.3984
3, linear: 1.07s | TVD=0.6334
1, full: 1.77s | TVD=0.0717
2, full: 1.12s | TVD=0.5643
3, full: 1.80s | TVD=0.3625


In [37]:
import time

# ------------------------------------------------------------
# PILOT SETTINGS
# ------------------------------------------------------------

PILOT_N_ROWS = 10
SHOTS = 1024
REPEATS = 5

CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

# These are our validated layouts
CANDIDATE_LAYOUTS = [
    [0, 1, 2, 3, 4],
    [19, 20, 21, 22, 23],
    [38, 39, 40, 41, 42],
    [57, 58, 59, 60, 61],
]

# ------------------------------------------------------------
# SAMPLE 10 DATA ROWS
# ------------------------------------------------------------

df_with_features = (
    df.dropna(subset=selected_columns)
      .reset_index(drop=True)
)

pilot_sample_df = (
    df_with_features
    .sample(
        n=PILOT_N_ROWS,
        random_state=42
    )
    .reset_index(drop=True)
)

print("Pilot sample shape:", pilot_sample_df.shape)

# ------------------------------------------------------------
# PREPARE RAW + SCALED FEATURES
# ------------------------------------------------------------

pilot_raw = (
    pilot_sample_df[selected_columns]
    .to_numpy()
)

pilot_scaled = scaler.transform(
    pilot_raw
)

# ------------------------------------------------------------
# TOTAL EXPERIMENT COUNT
# ------------------------------------------------------------

total_experiments = (
    PILOT_N_ROWS
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

print(
    f"\nTotal pilot experiments: {total_experiments}"
)
print(
    f"Expected noisy runs: "
    f"{total_experiments * REPEATS}"
)

# ------------------------------------------------------------
# RUN PILOT
# ------------------------------------------------------------

pilot_results = []

start = time.time()
experiment_number = 0

for row_idx in range(PILOT_N_ROWS):

    for reps, entanglement in CONFIGS:

        for layout in CANDIDATE_LAYOUTS:

            experiment_number += 1

            exp_start = time.time()

            result = run_complete_experiment(
                raw_data_row=pilot_raw[row_idx],
                data_row=pilot_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,

                shots=SHOTS,
                repeats=REPEATS,

                initial_layout=layout,
                seed_transpiler=42,
            )

            exp_time = time.time() - exp_start

            result["data_row_index"] = row_idx
            result["layout"] = str(layout)

            pilot_results.append(result)

            print(
                f"[{experiment_number}/{total_experiments}] "
                f"row={row_idx} | "
                f"reps={reps} | "
                f"entanglement={entanglement} | "
                f"layout={layout} | "
                f"time={exp_time:.2f}s | "
                f"TVD={result['mean_tvd']:.4f}"
            )

# ------------------------------------------------------------
# TIMING SUMMARY
# ------------------------------------------------------------

elapsed = time.time() - start

pilot_df = pd.DataFrame(pilot_results)

seconds_per_experiment = (
    elapsed / len(pilot_df)
)

FULL_N_ROWS = 200

full_experiments = (
    FULL_N_ROWS
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

estimated_full_seconds = (
    seconds_per_experiment
    * full_experiments
)

print("\n" + "=" * 60)
print("PILOT COMPLETE")
print("=" * 60)

print(
    f"Experiments completed : {len(pilot_df)}"
)

print(
    f"Total time            : "
    f"{elapsed / 60:.2f} min"
)

print(
    f"Average/experiment    : "
    f"{seconds_per_experiment:.2f} sec"
)

print(
    f"Estimated full dataset: "
    f"{estimated_full_seconds / 60:.1f} min"
)

print(
    f"Pilot dataframe shape : "
    f"{pilot_df.shape}"
)

Pilot sample shape: (10, 36)

Total pilot experiments: 240
Expected noisy runs: 1200
[1/240] row=0 | reps=1 | entanglement=linear | layout=[0, 1, 2, 3, 4] | time=1.61s | TVD=0.0688
[2/240] row=0 | reps=1 | entanglement=linear | layout=[19, 20, 21, 22, 23] | time=1.08s | TVD=0.0729
[3/240] row=0 | reps=1 | entanglement=linear | layout=[38, 39, 40, 41, 42] | time=1.62s | TVD=0.0744
[4/240] row=0 | reps=1 | entanglement=linear | layout=[57, 58, 59, 60, 61] | time=1.00s | TVD=0.0699
[5/240] row=0 | reps=2 | entanglement=linear | layout=[0, 1, 2, 3, 4] | time=1.62s | TVD=0.0905
[6/240] row=0 | reps=2 | entanglement=linear | layout=[19, 20, 21, 22, 23] | time=1.53s | TVD=0.4236
[7/240] row=0 | reps=2 | entanglement=linear | layout=[38, 39, 40, 41, 42] | time=1.58s | TVD=0.3041
[8/240] row=0 | reps=2 | entanglement=linear | layout=[57, 58, 59, 60, 61] | time=1.05s | TVD=0.4033
[9/240] row=0 | reps=3 | entanglement=linear | layout=[0, 1, 2, 3, 4] | time=1.62s | TVD=0.2467
[10/240] row=0 | reps

In [38]:
print("Shape:", pilot_df.shape)

print("\nMissing values:")
print(pilot_df.isna().sum().sort_values(ascending=False).head(15))
print("\nTarget statistics:")
print(
    pilot_df[
        ["mean_tvd", "std_tvd", "min_tvd", "max_tvd"]
    ].describe()
)
print("\nKey feature variation:")
print(
    pilot_df[
        [
            "reps",
            "entanglement",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "mean_tvd",
        ]
    ].groupby(
        ["reps", "entanglement"]
    ).mean()
)

Shape: (240, 61)

Missing values:
raw_feature_1                       0
raw_feature_2                       0
raw_feature_3                       0
raw_feature_4                       0
raw_feature_5                       0
angle_1                             0
angle_2                             0
angle_3                             0
angle_4                             0
angle_5                             0
abstract_n_qubits                   0
abstract_depth                      0
abstract_total_gate_count           0
abstract_single_qubit_gate_count    0
abstract_two_qubit_gate_count       0
dtype: int64

Target statistics:
         mean_tvd     std_tvd     min_tvd     max_tvd
count  240.000000  240.000000  240.000000  240.000000
mean     0.260870    0.011542    0.246559    0.275263
std      0.184390    0.004820    0.182299    0.186461
min      0.061328    0.003194    0.044922    0.066406
25%      0.072021    0.007981    0.063232    0.084717
50%      0.240377    0.011020    0.2216

In [39]:
target_columns = [
    "mean_tvd",
    "std_tvd",
    "min_tvd",
    "max_tvd"
]

X_pilot = pilot_df.drop(columns=target_columns)
y_pilot = pilot_df["mean_tvd"]

print("Predictors:", X_pilot.shape)
print("Target:", y_pilot.shape)

print("\nDuplicate columns:")
print(
    X_pilot.columns[
        X_pilot.T.duplicated()
    ].tolist()
)

Predictors: (240, 57)
Target: (240,)

Duplicate columns:
['abstract_cx_count', 'transpiled_cz_count', 'physical_qubit_count', 'repeats', 'initial_layout', 'layout']


In [40]:
DROP_FROM_MODEL = [
    "physical_qubit_count",
    "repeats",
    "layout",
]

X_pilot_clean = X_pilot.drop(
    columns=DROP_FROM_MODEL
)

print("Original predictors:", X_pilot.shape)
print("Clean predictors:", X_pilot_clean.shape)
print("\nRemaining columns:")
print(X_pilot_clean.columns.tolist())

Original predictors: (240, 57)
Clean predictors: (240, 54)

Remaining columns:
['raw_feature_1', 'raw_feature_2', 'raw_feature_3', 'raw_feature_4', 'raw_feature_5', 'angle_1', 'angle_2', 'angle_3', 'angle_4', 'angle_5', 'abstract_n_qubits', 'abstract_depth', 'abstract_total_gate_count', 'abstract_single_qubit_gate_count', 'abstract_two_qubit_gate_count', 'abstract_cx_count', 'abstract_two_qubit_gate_ratio', 'abstract_average_gates_per_qubit', 'abstract_max_gates_per_qubit', 'abstract_unique_2q_pairs', 'abstract_average_qubit_degree', 'abstract_max_qubit_degree', 'abstract_repeated_2q_pairs', 'reps', 'entanglement', 'transpiled_depth', 'transpiled_total_gates', 'transpiled_1q_gates', 'transpiled_2q_gates', 'transpiled_cz_count', 'transpiled_cx_count', 'transpiled_sx_count', 'transpiled_rz_count', 'backend_size', 'mean_cz_error', 'std_cz_error', 'min_cz_error', 'max_cz_error', 'sum_cz_error', 'cumulative_cz_error', 'mean_cz_duration', 'max_cz_duration', 'total_cz_duration', 'physical_qub

In [41]:
pilot_df.to_csv(
    "../experiments/results/pilot_240.csv",
    index=False
)

print("Saved pilot:", pilot_df.shape)

Saved pilot: (240, 61)


In [42]:
PRIMARY_TARGET = "mean_tvd"

TARGET_COLUMNS = [
    "mean_tvd",
    "std_tvd",
    "min_tvd",
    "max_tvd",
]

In [43]:
MODEL_EXCLUDE = [
    "mean_tvd",
    "std_tvd",
    "min_tvd",
    "max_tvd",
    "physical_qubit_count",
    "repeats",
    "layout",
    "data_row_index",
    "shots",
    "seed_transpiler",
]

### Full dataset generation

**Only run this after checking the pilot timing above is acceptable.**
Every (row, config, layout) combination is run independently -- layout
is fully crossed with data and configuration, not tied to row index.

In [45]:
from sklearn.model_selection import train_test_split
N_ROWS = 200

sampled_df, _ = train_test_split(
    df_with_features,
    train_size=N_ROWS,
    stratify=df_with_features["cell_type"],
    random_state=42,
)

sampled_raw = sampled_df[selected_columns].values
sampled_scaled = scaler.transform(sampled_raw)

print("Sampled rows:", sampled_raw.shape)
print(
    "Total experiments to run:",
    len(sampled_raw) * len(CONFIGS) * len(CANDIDATE_LAYOUTS)
)

Sampled rows: (200, 5)
Total experiments to run: 4800


In [48]:
print(type(noisy_simulator))
print(CANDIDATE_LAYOUTS)
print(len(sampled_raw))

<class 'qiskit_aer.backends.aer_simulator.AerSimulator'>
[[0, 1, 2, 3, 4], [19, 20, 21, 22, 23], [38, 39, 40, 41, 42], [57, 58, 59, 60, 61]]
200


In [49]:
all_results = []

total = (
    len(sampled_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

done = 0
start_time = time.time()

print(f"Starting full dataset generation...")
print(f"Rows: {len(sampled_raw)}")
print(f"Configurations: {len(CONFIGS)}")
print(f"Layouts: {len(CANDIDATE_LAYOUTS)}")
print(f"Total experiments: {total}")
print(f"Repeats per experiment: 5")

for row_idx in range(len(sampled_raw)):

    for reps, entanglement in CONFIGS:

        for layout in CANDIDATE_LAYOUTS:

            result = run_complete_experiment(
                raw_data_row=sampled_raw[row_idx],
                data_row=sampled_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            result["data_row_index"] = row_idx
            result["layout_id"] = str(layout)

            all_results.append(result)

            done += 1

    if (row_idx + 1) % 10 == 0:
        elapsed = time.time() - start_time

        print(
            f"Completed {row_idx + 1}/{len(sampled_raw)} rows "
            f"({done}/{total} experiments) | "
            f"Elapsed: {elapsed/60:.1f} min"
        )

predictor_df = pd.DataFrame(all_results)

print("\nFinal dataset shape:", predictor_df.shape)

Starting full dataset generation...
Rows: 200
Configurations: 6
Layouts: 4
Total experiments: 4800
Repeats per experiment: 5
Completed 10/200 rows (240/4800 experiments) | Elapsed: 17.1 min
Completed 20/200 rows (480/4800 experiments) | Elapsed: 35.1 min
Completed 30/200 rows (720/4800 experiments) | Elapsed: 53.5 min
Completed 40/200 rows (960/4800 experiments) | Elapsed: 71.1 min
Completed 50/200 rows (1200/4800 experiments) | Elapsed: 89.1 min
Completed 60/200 rows (1440/4800 experiments) | Elapsed: 107.5 min
Completed 70/200 rows (1680/4800 experiments) | Elapsed: 125.7 min
Completed 80/200 rows (1920/4800 experiments) | Elapsed: 143.5 min
Completed 90/200 rows (2160/4800 experiments) | Elapsed: 161.8 min
Completed 100/200 rows (2400/4800 experiments) | Elapsed: 178.4 min
Completed 110/200 rows (2640/4800 experiments) | Elapsed: 193.3 min
Completed 120/200 rows (2880/4800 experiments) | Elapsed: 201.9 min
Completed 130/200 rows (3120/4800 experiments) | Elapsed: 210.6 min
Completed

In [50]:
predictor_df.to_csv("../experiments/results/predictor_training_data.csv", index=False)
print("Saved:", predictor_df.shape)


Saved: (4800, 61)


In [51]:
print("Shape:", predictor_df.shape)

print("\nMissing values:")
print(
    predictor_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Shape: (4800, 61)

Missing values:
raw_feature_1                       0
raw_feature_2                       0
raw_feature_3                       0
raw_feature_4                       0
raw_feature_5                       0
angle_1                             0
angle_2                             0
angle_3                             0
angle_4                             0
angle_5                             0
abstract_n_qubits                   0
abstract_depth                      0
abstract_total_gate_count           0
abstract_single_qubit_gate_count    0
abstract_two_qubit_gate_count       0
abstract_cx_count                   0
abstract_two_qubit_gate_ratio       0
abstract_average_gates_per_qubit    0
abstract_max_gates_per_qubit        0
abstract_unique_2q_pairs            0
dtype: int64


In [52]:
print("\nDuplicate rows:", predictor_df.duplicated().sum())


Duplicate rows: 0


In [53]:
print("\nTVD statistics:")
print(
    predictor_df[
        ["mean_tvd", "std_tvd", "min_tvd", "max_tvd"]
    ].describe()
)


TVD statistics:
          mean_tvd      std_tvd      min_tvd      max_tvd
count  4800.000000  4800.000000  4800.000000  4800.000000
mean      0.255241     0.011667     0.240840     0.269728
std       0.170379     0.004798     0.168411     0.172390
min       0.055273     0.000764     0.039063     0.060547
25%       0.072070     0.008142     0.062500     0.084961
50%       0.241873     0.011119     0.225840     0.260251
75%       0.388863     0.014697     0.372669     0.405674
max       0.759791     0.034468     0.753736     0.767408


In [54]:
print(predictor_df.columns.tolist())

['raw_feature_1', 'raw_feature_2', 'raw_feature_3', 'raw_feature_4', 'raw_feature_5', 'angle_1', 'angle_2', 'angle_3', 'angle_4', 'angle_5', 'abstract_n_qubits', 'abstract_depth', 'abstract_total_gate_count', 'abstract_single_qubit_gate_count', 'abstract_two_qubit_gate_count', 'abstract_cx_count', 'abstract_two_qubit_gate_ratio', 'abstract_average_gates_per_qubit', 'abstract_max_gates_per_qubit', 'abstract_unique_2q_pairs', 'abstract_average_qubit_degree', 'abstract_max_qubit_degree', 'abstract_repeated_2q_pairs', 'reps', 'entanglement', 'transpiled_depth', 'transpiled_total_gates', 'transpiled_1q_gates', 'transpiled_2q_gates', 'transpiled_cz_count', 'transpiled_cx_count', 'transpiled_sx_count', 'transpiled_rz_count', 'backend_size', 'mean_cz_error', 'std_cz_error', 'min_cz_error', 'max_cz_error', 'sum_cz_error', 'cumulative_cz_error', 'mean_cz_duration', 'max_cz_duration', 'total_cz_duration', 'physical_qubit_count', 'physical_qubits_used', 'mean_t1', 'min_t1', 'max_t1', 'mean_t2', 'm

In [55]:
import pandas as pd
import numpy as np

# Work on a copy
validation_df = predictor_df.copy()

# ------------------------------------------------------------
# 1. Create 3 TVD bands
# ------------------------------------------------------------

validation_df["tvd_band"] = pd.qcut(
    validation_df["mean_tvd"],
    q=3,
    labels=["low", "medium", "high"]
)

# ------------------------------------------------------------
# 2. We want diversity across:
#    - reps
#    - entanglement
#    - hardware layout
# ------------------------------------------------------------

selected_rows = []

used_combinations = set()

for band in ["low", "medium", "high"]:

    candidates = (
        validation_df[
            validation_df["tvd_band"] == band
        ]
        .sort_values("mean_tvd")
    )

    # Try to select 4 diverse circuits from this band
    for _, row in candidates.iterrows():

        combo = (
            row["reps"],
            row["entanglement"],
            row["layout_id"]
        )

        if combo not in used_combinations:
            selected_rows.append(row)
            used_combinations.add(combo)

        if len([
            r for r in selected_rows
            if r["tvd_band"] == band
        ]) >= 4:
            break

# ------------------------------------------------------------
# 3. Create final validation dataframe
# ------------------------------------------------------------

hardware_validation_df = pd.DataFrame(
    selected_rows
).reset_index(drop=True)

print(
    "Validation circuits:",
    len(hardware_validation_df)
)

print(
    hardware_validation_df[
        [
            "data_row_index",
            "reps",
            "entanglement",
            "layout_id",
            "initial_layout",
            "mean_tvd",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "tvd_band"
        ]
    ]
)

Validation circuits: 12
    data_row_index  reps entanglement             layout_id        initial_layout  mean_tvd  transpiled_depth  \
0              184     1         full  [19, 20, 21, 22, 23]  [19, 20, 21, 22, 23]  0.055273               125   
1                8     1       linear  [38, 39, 40, 41, 42]  [38, 39, 40, 41, 42]  0.055664                43   
2              122     1         full       [0, 1, 2, 3, 4]       [0, 1, 2, 3, 4]  0.056250               123   
3              180     1       linear  [19, 20, 21, 22, 23]  [19, 20, 21, 22, 23]  0.057031                45   
4               60     2       linear       [0, 1, 2, 3, 4]       [0, 1, 2, 3, 4]  0.083388                67   
5                6     3       linear       [0, 1, 2, 3, 4]       [0, 1, 2, 3, 4]  0.083523                89   
6               60     3       linear  [38, 39, 40, 41, 42]  [38, 39, 40, 41, 42]  0.093295                89   
7               58     2       linear  [38, 39, 40, 41, 42]  [38, 39, 40

In [56]:
print("\nConfiguration coverage:")
print(
    hardware_validation_df[
        ["reps", "entanglement"]
    ].value_counts()
)

print("\nLayout coverage:")
print(
    hardware_validation_df["layout_id"].value_counts()
)

print("\nTVD bands:")
print(
    hardware_validation_df["tvd_band"].value_counts()
)


Configuration coverage:
reps  entanglement
3     linear          3
1     full            2
      linear          2
2     linear          2
3     full            2
2     full            1
Name: count, dtype: int64

Layout coverage:
layout_id
[19, 20, 21, 22, 23]    4
[38, 39, 40, 41, 42]    4
[0, 1, 2, 3, 4]         4
Name: count, dtype: int64

TVD bands:
tvd_band
low       4
medium    4
high      4
Name: count, dtype: int64


In [57]:
row = hardware_validation_df.iloc[0]

print("Layout:", row["initial_layout"])
print("Physical qubits:", row["physical_qubits_used"])
print("Stored mean CZ error:", row["mean_cz_error"])

Layout: [19, 20, 21, 22, 23]
Physical qubits: [19, 20, 21, 22, 23]
Stored mean CZ error: 0.2681439247659132


In [58]:
# Recreate the same circuit
raw_row = df_with_features.iloc[
    int(row["data_row_index"])
][selected_columns].to_numpy()

scaled_row = scaler.transform(
    raw_row.reshape(1, -1)
)[0]

fm = zz_feature_map(
    feature_dimension=5,
    reps=int(row["reps"]),
    entanglement=row["entanglement"]
)

abstract = fm.assign_parameters(scaled_row)

measured = abstract.copy()
measured.measure_all()

transpiled = transpile(
    measured,
    fake_backend,
    optimization_level=1,
    initial_layout=eval(row["initial_layout"]),
    seed_transpiler=42,
)

print("\nActual physical 2Q operations:")

for inst in transpiled.data:
    if inst.operation.name in ["cz", "cx"]:
        qubits = [
            transpiled.find_bit(q).index
            for q in inst.qubits
        ]
        print(inst.operation.name, qubits)


Actual physical 2Q operations:
cz [19, 20]
cz [19, 20]
cz [20, 21]
cz [20, 21]
cz [20, 21]
cz [19, 20]
cz [19, 20]
cz [21, 20]
cz [21, 20]
cz [20, 19]
cz [20, 19]
cz [20, 19]
cz [21, 20]
cz [21, 20]
cz [21, 20]
cz [21, 22]
cz [21, 22]
cz [22, 21]
cz [22, 21]
cz [22, 21]
cz [20, 21]
cz [20, 21]
cz [21, 20]
cz [21, 20]
cz [21, 20]
cz [19, 20]
cz [19, 20]
cz [22, 23]
cz [22, 23]
cz [22, 23]
cz [22, 23]
cz [22, 23]
cz [21, 22]
cz [21, 22]
cz [21, 22]
cz [21, 22]
cz [21, 22]
cz [20, 21]
cz [20, 21]
cz [20, 21]
cz [19, 20]
cz [19, 20]
cz [21, 20]
cz [21, 20]


In [59]:
pairs = [
    (19, 20),
    (20, 21),
    (21, 22),
    (22, 23),
]

edges = {
    tuple(sorted(edge))
    for edge in fake_backend.coupling_map.get_edges()
}

print("Backend calibration for used CZ pairs:\n")

for q1, q2 in pairs:
    pair = tuple(sorted((q1, q2)))

    print(f"\nPair {pair}")

    if pair not in edges:
        print("  NOT FOUND in coupling map")
        continue

    # Check both directions because backend properties may be directional
    for direction in [(q1, q2), (q2, q1)]:
        try:
            props = fake_backend.target["cz"][
                direction
            ]

            print(
                f"  {direction}: "
                f"error={props.error}, "
                f"duration={props.duration}"
            )

        except Exception as e:
            print(
                f"  {direction}: "
                f"could not retrieve target CZ properties: {e}"
            )

Backend calibration for used CZ pairs:


Pair (19, 20)
  (19, 20): error=1.0, duration=1.08e-07
  (20, 19): error=1.0, duration=1.08e-07

Pair (20, 21)
  (20, 21): error=0.03544601345484144, duration=6.8e-08
  (21, 20): error=0.03544601345484144, duration=6.8e-08

Pair (21, 22)
  (21, 22): error=0.012872397774247424, duration=6.8e-08
  (22, 21): error=0.012872397774247424, duration=6.8e-08

Pair (22, 23)
  (22, 23): error=0.006316093954111535, duration=6.8e-08
  (23, 22): error=0.006316093954111535, duration=6.8e-08


In [60]:
for layout in CANDIDATE_LAYOUTS:
    print("\nLayout:", layout)

    for i in range(len(layout) - 1):
        q1, q2 = layout[i], layout[i+1]

        try:
            props = fake_backend.target["cz"][(q1, q2)]
            print(
                f"  ({q1},{q2}) "
                f"error={props.error:.6f}, "
                f"duration={props.duration:.2e}s"
            )
        except Exception:
            props = fake_backend.target["cz"][(q2, q1)]
            print(
                f"  ({q2},{q1}) "
                f"error={props.error:.6f}, "
                f"duration={props.duration:.2e}s"
            )


Layout: [0, 1, 2, 3, 4]
  (0,1) error=0.004855, duration=6.80e-08s
  (1,2) error=0.004194, duration=6.80e-08s
  (2,3) error=0.003561, duration=6.80e-08s
  (3,4) error=0.004260, duration=6.80e-08s

Layout: [19, 20, 21, 22, 23]
  (19,20) error=1.000000, duration=1.08e-07s
  (20,21) error=0.035446, duration=6.80e-08s
  (21,22) error=0.012872, duration=6.80e-08s
  (22,23) error=0.006316, duration=6.80e-08s

Layout: [38, 39, 40, 41, 42]
  (38,39) error=0.002595, duration=6.80e-08s
  (39,40) error=0.004605, duration=6.80e-08s
  (40,41) error=0.064664, duration=6.80e-08s
  (41,42) error=0.071514, duration=6.80e-08s

Layout: [57, 58, 59, 60, 61]
  (57,58) error=1.000000, duration=6.80e-08s
  (58,59) error=1.000000, duration=6.80e-08s
  (59,60) error=0.003291, duration=6.80e-08s
  (60,61) error=0.002720, duration=6.80e-08s


In [61]:
def layout_edge_errors(layout):
    errors = []

    for i in range(len(layout) - 1):
        q1, q2 = layout[i], layout[i+1]

        try:
            props = fake_backend.target["cz"][(q1, q2)]
        except Exception:
            props = fake_backend.target["cz"][(q2, q1)]

        errors.append(props.error)

    return errors


candidate_quality = []

for layout in valid_layouts:
    errors = layout_edge_errors(layout)

    candidate_quality.append({
        "layout": layout,
        "max_error": max(errors),
        "mean_error": sum(errors) / len(errors),
        "errors": errors
    })

# Remove obviously broken layouts
good_layouts = [
    x for x in candidate_quality
    if x["max_error"] < 0.1
]

# Sort by mean error
good_layouts = sorted(
    good_layouts,
    key=lambda x: x["mean_error"]
)

print("Good candidate layouts:")
for x in good_layouts[:20]:
    print(
        x["layout"],
        "mean=", round(x["mean_error"], 6),
        "max=", round(x["max_error"], 6),
        "edges=", [round(e, 6) for e in x["errors"]]
    )

Good candidate layouts:
[63, 64, 65, 66, 67] mean= 0.002881 max= 0.003303 edges= [0.003089, 0.002328, 0.002804, 0.003303]
[64, 65, 66, 67, 68] mean= 0.002883 max= 0.003303 edges= [0.002328, 0.002804, 0.003303, 0.003097]
[62, 63, 64, 65, 66] mean= 0.003267 max= 0.004847 edges= [0.004847, 0.003089, 0.002328, 0.002804]
[65, 66, 67, 68, 69] mean= 0.003283 max= 0.003929 edges= [0.002804, 0.003303, 0.003097, 0.003929]
[23, 24, 25, 26, 27] mean= 0.00333 max= 0.004405 edges= [0.003167, 0.002954, 0.002793, 0.004405]
[61, 62, 63, 64, 65] mean= 0.003418 max= 0.004847 edges= [0.003406, 0.004847, 0.003089, 0.002328]
[60, 61, 62, 63, 64] mean= 0.003515 max= 0.004847 edges= [0.00272, 0.003406, 0.004847, 0.003089]
[59, 60, 61, 62, 63] mean= 0.003566 max= 0.004847 edges= [0.003291, 0.00272, 0.003406, 0.004847]
[9, 10, 11, 12, 13] mean= 0.003639 max= 0.004138 edges= [0.004072, 0.003466, 0.002879, 0.004138]
[98, 99, 100, 101, 102] mean= 0.003651 max= 0.004187 edges= [0.002537, 0.004051, 0.003828, 0.00418

In [65]:
validation_df = predictor_df.copy()

validation_df["tvd_band"] = pd.qcut(
    validation_df["mean_tvd"],
    q=3,
    labels=["low", "medium", "high"]
)

# Pick 4 rows from each band
low = validation_df[validation_df["tvd_band"] == "low"].sample(
    n=4, random_state=42
)

medium = validation_df[validation_df["tvd_band"] == "medium"].sample(
    n=4, random_state=42
)

high = validation_df[validation_df["tvd_band"] == "high"].sample(
    n=4, random_state=42
)

hardware_validation_df = pd.concat(
    [low, medium, high],
    ignore_index=True
)

print("Validation circuits:", len(hardware_validation_df))

print(
    hardware_validation_df[
        [
            "data_row_index",
            "reps",
            "entanglement",
            "initial_layout",
            "mean_tvd",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "tvd_band"
        ]
    ]
)

Validation circuits: 12
    data_row_index  reps entanglement        initial_layout  mean_tvd  transpiled_depth  transpiled_2q_gates  \
0               65     1         full  [57, 58, 59, 60, 61]  0.065820               125                   44   
1               44     1       linear  [57, 58, 59, 60, 61]  0.070313                45                    8   
2               21     1       linear  [19, 20, 21, 22, 23]  0.069336                45                    8   
3               17     1       linear       [0, 1, 2, 3, 4]  0.072852                43                    8   
4               67     2         full       [0, 1, 2, 3, 4]  0.195166               240                   91   
5               46     3       linear       [0, 1, 2, 3, 4]  0.164342                87                   24   
6               21     3       linear  [57, 58, 59, 60, 61]  0.331858                89                   24   
7               17     3       linear  [38, 39, 40, 41, 42]  0.298367           

In [66]:
hardware_validation_df.to_csv(
    "../experiments/results/hardware_validation_12.csv",
    index=False
)

print("Saved:", hardware_validation_df.shape)

Saved: (12, 62)


In [68]:
import ast

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    raw_row = sampled_raw[data_idx]
    scaled_row = sampled_scaled[data_idx]

    reps = int(row["reps"])
    entanglement = row["entanglement"]
    layout = ast.literal_eval(row["initial_layout"])

    fm = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement
    )

    abstract = fm.assign_parameters(scaled_row)

    measured = abstract.copy()
    measured.measure_all()

    transpiled = transpile(
        measured,
        fake_backend,
        optimization_level=1,
        initial_layout=layout,
        seed_transpiler=42
    )

    recreated_2q = sum(
        1
        for inst in transpiled.data
        if len(inst.qubits) == 2
    )

    print(
        f"{i}: "
        f"stored depth={row['transpiled_depth']}, "
        f"recreated depth={transpiled.depth()} | "
        f"stored 2Q={row['transpiled_2q_gates']}, "
        f"recreated 2Q={recreated_2q}"
    )

0: stored depth=125, recreated depth=126 | stored 2Q=44, recreated 2Q=44
1: stored depth=45, recreated depth=46 | stored 2Q=8, recreated 2Q=8
2: stored depth=45, recreated depth=46 | stored 2Q=8, recreated 2Q=8
3: stored depth=43, recreated depth=44 | stored 2Q=8, recreated 2Q=8
4: stored depth=240, recreated depth=241 | stored 2Q=91, recreated 2Q=91
5: stored depth=87, recreated depth=88 | stored 2Q=24, recreated 2Q=24
6: stored depth=89, recreated depth=90 | stored 2Q=24, recreated 2Q=24
7: stored depth=87, recreated depth=88 | stored 2Q=24, recreated 2Q=24
8: stored depth=67, recreated depth=68 | stored 2Q=16, recreated 2Q=16
9: stored depth=233, recreated depth=234 | stored 2Q=91, recreated 2Q=91
10: stored depth=233, recreated depth=234 | stored 2Q=91, recreated 2Q=91
11: stored depth=353, recreated depth=354 | stored 2Q=138, recreated 2Q=138


In [ ]:
"""
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="",
    instance="",
    set_as_default=True,
    overwrite=True,
)

print("Credentials saved.")
"""

Credentials saved.


In [ ]:
from dotenv import load_dotenv
import os
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    token=os.environ["IBM_QUANTUM_TOKEN"],
    instance=os.environ["IBM_QUANTUM_INSTANCE"],
)

In [71]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

print("IBM Quantum connection OK")
print(service.backends())

IBM Quantum connection OK
[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


In [72]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

backend = service.backend("ibm_kingston")

print("Backend:", backend.name)
print("Qubits:", backend.num_qubits)
print("Processor:", backend.processor_type)
print("Basis gates:", backend.operation_names)

Backend: ibm_kingston
Qubits: 156
Processor: {'family': 'Heron', 'revision': '2'}
Basis gates: ['sx', 'measure', 'rz', 'reset', 'if_else', 'delay', 'measure_2', 'x', 'cz', 'id']


In [73]:
print("Number of qubits:", backend.num_qubits)

edges = backend.coupling_map.get_edges()

print("Number of coupling edges:", len(edges))
print("First 30 coupling edges:")
for edge in edges[:30]:
    print(edge)

Number of qubits: 156
Number of coupling edges: 352
First 30 coupling edges:
(0, 1)
(1, 0)
(1, 2)
(2, 1)
(2, 3)
(3, 2)
(3, 4)
(3, 16)
(4, 3)
(4, 5)
(5, 4)
(5, 6)
(6, 5)
(6, 7)
(7, 6)
(7, 8)
(7, 17)
(8, 7)
(8, 9)
(9, 8)
(9, 10)
(10, 9)
(10, 11)
(11, 10)
(11, 12)
(11, 18)
(12, 11)
(12, 13)
(13, 12)
(13, 14)


In [74]:
# Look for connected 5-qubit chains
coupling_edges = {
    tuple(sorted(edge))
    for edge in backend.coupling_map.get_edges()
}

kingston_chains = []

for start in range(backend.num_qubits - 4):
    layout = list(range(start, start + 5))

    connected = all(
        tuple(sorted((layout[i], layout[i + 1]))) in coupling_edges
        for i in range(4)
    )

    if connected:
        kingston_chains.append(layout)

print("Connected 5-qubit chains found:", len(kingston_chains))

for layout in kingston_chains[:20]:
    print(layout)

Connected 5-qubit chains found: 96
[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[3, 4, 5, 6, 7]
[4, 5, 6, 7, 8]
[5, 6, 7, 8, 9]
[6, 7, 8, 9, 10]
[7, 8, 9, 10, 11]
[8, 9, 10, 11, 12]
[9, 10, 11, 12, 13]
[10, 11, 12, 13, 14]
[11, 12, 13, 14, 15]
[20, 21, 22, 23, 24]
[21, 22, 23, 24, 25]
[22, 23, 24, 25, 26]
[23, 24, 25, 26, 27]
[24, 25, 26, 27, 28]
[25, 26, 27, 28, 29]
[26, 27, 28, 29, 30]
[27, 28, 29, 30, 31]


In [75]:
from qiskit_ibm_runtime.fake_provider import FakeKingston

fake_kingston = FakeKingston()

print("Fake backend:", fake_kingston.name)
print("Qubits:", fake_kingston.num_qubits)
print("Basis gates:", fake_kingston.operation_names)

Fake backend: fake_kingston
Qubits: 156
Basis gates: ['sx', 'measure', 'rz', 'reset', 'if_else', 'delay', 'measure_2', 'x', 'cz', 'id']


In [76]:
KINGSTON_VALIDATION_LAYOUT = [0, 1, 2, 3, 4]

print("Kingston validation layout:", KINGSTON_VALIDATION_LAYOUT)

Kingston validation layout: [0, 1, 2, 3, 4]


In [77]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

kingston_noise_model = NoiseModel.from_backend(fake_kingston)

fake_kingston_sim = AerSimulator(
    noise_model=kingston_noise_model
)

print("FakeKingston noisy simulator ready.")

FakeKingston noisy simulator ready.


In [78]:
import ast
import numpy as np
import pandas as pd
from qiskit import transpile
from qiskit.quantum_info import Statevector

fake_kingston_results = []

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    raw_row = sampled_raw[data_idx]
    scaled_row = sampled_scaled[data_idx]

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    # --------------------------------------------------------
    # Recreate SAME logical circuit
    # --------------------------------------------------------

    fm = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = fm.assign_parameters(
        scaled_row
    )

    # Exact ideal distribution
    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    # Add measurements
    measured = abstract_circuit.copy()
    measured.measure_all()

    # --------------------------------------------------------
    # Transpile for FakeKingston
    # --------------------------------------------------------

    kingston_circuit = transpile(
        measured,
        fake_kingston,
        optimization_level=1,
        initial_layout=KINGSTON_VALIDATION_LAYOUT,
        seed_transpiler=42,
    )

    # --------------------------------------------------------
    # Run 5 noisy repetitions in one job
    # --------------------------------------------------------

    circuits = [
        kingston_circuit
        for _ in range(5)
    ]

    result = fake_kingston_sim.run(
        circuits,
        shots=1024,
    ).result()

    tvds = []

    for j in range(5):

        counts = result.get_counts(j)

        noisy_probs = counts_to_probabilities(
            counts
        )

        tvd = calculate_tvd_from_probs(
            ideal_probs,
            noisy_probs
        )

        tvds.append(tvd)

    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    fake_kingston_results.append({
        "validation_index": i,
        "data_row_index": data_idx,
        "reps": reps,
        "entanglement": entanglement,

        # Original Torino result
        "torino_mean_tvd": row["mean_tvd"],

        # Kingston simulation
        "kingston_mean_tvd": float(np.mean(tvds)),
        "kingston_std_tvd": float(
            np.std(tvds, ddof=1)
        ),

        "kingston_depth": kingston_circuit.depth(),

        "kingston_2q_gates": sum(
            1
            for inst in kingston_circuit.data
            if len(inst.qubits) == 2
        ),

        "kingston_layout": str(
            KINGSTON_VALIDATION_LAYOUT
        ),
    })

fake_kingston_validation_df = pd.DataFrame(
    fake_kingston_results
)

print(
    fake_kingston_validation_df
)

    validation_index  data_row_index  reps entanglement  torino_mean_tvd  kingston_mean_tvd  kingston_std_tvd  \
0                  0              65     1         full         0.065820           0.076953          0.005584   
1                  1              44     1       linear         0.070313           0.066797          0.004863   
2                  2              21     1       linear         0.069336           0.063867          0.013612   
3                  3              17     1       linear         0.072852           0.067578          0.007640   
4                  4              67     2         full         0.195166           0.127191          0.005798   
5                  5              46     3       linear         0.164342           0.099062          0.010104   
6                  6              21     3       linear         0.331858           0.110679          0.010438   
7                  7              17     3       linear         0.298367           0.119019     

In [79]:
print(
    fake_kingston_validation_df[
        [
            "validation_index",
            "reps",
            "entanglement",
            "torino_mean_tvd",
            "kingston_mean_tvd",
        ]
    ]
)

    validation_index  reps entanglement  torino_mean_tvd  kingston_mean_tvd
0                  0     1         full         0.065820           0.076953
1                  1     1       linear         0.070313           0.066797
2                  2     1       linear         0.069336           0.063867
3                  3     1       linear         0.072852           0.067578
4                  4     2         full         0.195166           0.127191
5                  5     3       linear         0.164342           0.099062
6                  6     3       linear         0.331858           0.110679
7                  7     3       linear         0.298367           0.119019
8                  8     2       linear         0.530607           0.087640
9                  9     2         full         0.361257           0.164500
10                10     2         full         0.371899           0.151002
11                11     3         full         0.362996           0.136236


In [80]:
corr = fake_kingston_validation_df[
    ["torino_mean_tvd", "kingston_mean_tvd"]
].corr().iloc[0, 1]

mae = np.mean(
    np.abs(
        fake_kingston_validation_df["torino_mean_tvd"]
        -
        fake_kingston_validation_df["kingston_mean_tvd"]
    )
)

print(f"Correlation: {corr:.4f}")
print(f"MAE: {mae:.4f}")

Correlation: 0.6356
MAE: 0.1372


In [81]:
fake_kingston_validation_df.to_csv(
    "../experiments/results/fake_kingston_validation_12.csv",
    index=False
)

print("Saved:", fake_kingston_validation_df.shape)

Saved: (12, 10)


In [82]:
KINGSTON_LAYOUT = [0, 1, 2, 3, 4]
real_kingston_circuits = []

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    scaled_row = sampled_scaled[data_idx]

    fm = zz_feature_map(
        feature_dimension=5,
        reps=int(row["reps"]),
        entanglement=row["entanglement"],
    )

    abstract = fm.assign_parameters(scaled_row)

    measured = abstract.copy()
    measured.measure_all()

    isa_circuit = transpile(
        measured,
        backend,
        optimization_level=1,
        initial_layout=[0, 1, 2, 3, 4],
        seed_transpiler=42,
    )

    real_kingston_circuits.append(isa_circuit)

    print(
        f"{i}: "
        f"reps={row['reps']} | "
        f"entanglement={row['entanglement']} | "
        f"depth={isa_circuit.depth()} | "
        f"2Q={sum(len(inst.qubits) == 2 for inst in isa_circuit.data)}"
    )

0: reps=1 | entanglement=full | depth=126 | 2Q=44
1: reps=1 | entanglement=linear | depth=46 | 2Q=8
2: reps=1 | entanglement=linear | depth=46 | 2Q=8
3: reps=1 | entanglement=linear | depth=44 | 2Q=8
4: reps=2 | entanglement=full | depth=241 | 2Q=91
5: reps=3 | entanglement=linear | depth=88 | 2Q=24
6: reps=3 | entanglement=linear | depth=90 | 2Q=24
7: reps=3 | entanglement=linear | depth=88 | 2Q=24
8: reps=2 | entanglement=linear | depth=68 | 2Q=16
9: reps=2 | entanglement=full | depth=234 | 2Q=91
10: reps=2 | entanglement=full | depth=234 | 2Q=91
11: reps=3 | entanglement=full | depth=354 | 2Q=138


In [84]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

job = sampler.run(
    real_kingston_circuits,
    shots=1024
)

print("Job ID:", job.job_id())
print("Submitted to:", backend.name)

Job ID: da0btmv2sl0c73bmtntg
Submitted to: ibm_kingston


In [85]:
result = job.result()

print("Job completed.")
print("Number of circuit results:", len(result))

Job completed.
Number of circuit results: 12


In [86]:
print(result[0])

SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}})


In [87]:
real_kingston_counts = []

for i, pub_result in enumerate(result):

    counts = pub_result.data.meas.get_counts()

    real_kingston_counts.append(counts)

    print(f"\nCircuit {i}")
    print(counts)


Circuit 0
{'10010': 24, '01010': 36, '00001': 32, '00100': 40, '11111': 16, '11011': 32, '10001': 35, '11000': 39, '01011': 34, '00111': 30, '11001': 38, '10110': 28, '01001': 36, '10000': 49, '01101': 27, '10100': 37, '11010': 40, '10101': 24, '00010': 38, '11110': 22, '00011': 40, '00110': 36, '00000': 28, '01100': 33, '11100': 41, '01000': 41, '11101': 29, '01111': 22, '01110': 17, '00101': 33, '10011': 26, '10111': 21}

Circuit 1
{'01000': 26, '10011': 30, '00100': 32, '01010': 27, '01111': 36, '01001': 34, '11100': 34, '01011': 34, '10010': 43, '01101': 25, '01100': 21, '10100': 38, '10000': 39, '00010': 26, '00011': 27, '00000': 46, '00111': 36, '00101': 32, '11111': 38, '00001': 31, '11010': 34, '11000': 52, '11001': 33, '11110': 27, '10001': 31, '10111': 27, '11011': 31, '01110': 23, '11101': 37, '00110': 28, '10110': 22, '10101': 24}

Circuit 2
{'00101': 38, '00000': 37, '00010': 37, '10001': 36, '11000': 21, '10000': 31, '10110': 36, '11001': 28, '10111': 35, '00001': 37, '1

In [88]:
for i, counts in enumerate(real_kingston_counts):
    print(
        f"Circuit {i}: "
        f"{sum(counts.values())} shots, "
        f"{len(counts)} unique outcomes"
    )

Circuit 0: 1024 shots, 32 unique outcomes
Circuit 1: 1024 shots, 32 unique outcomes
Circuit 2: 1024 shots, 32 unique outcomes
Circuit 3: 1024 shots, 32 unique outcomes
Circuit 4: 1024 shots, 32 unique outcomes
Circuit 5: 1024 shots, 32 unique outcomes
Circuit 6: 1024 shots, 31 unique outcomes
Circuit 7: 1024 shots, 30 unique outcomes
Circuit 8: 1024 shots, 32 unique outcomes
Circuit 9: 1024 shots, 32 unique outcomes
Circuit 10: 1024 shots, 32 unique outcomes
Circuit 11: 1024 shots, 32 unique outcomes


In [89]:
from qiskit.quantum_info import Statevector

real_validation_results = []

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    scaled_row = sampled_scaled[data_idx]

    fm = zz_feature_map(
        feature_dimension=5,
        reps=int(row["reps"]),
        entanglement=row["entanglement"],
    )

    abstract_circuit = fm.assign_parameters(
        scaled_row
    )

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    hardware_counts = real_kingston_counts[i]

    hardware_probs = counts_to_probabilities(
        hardware_counts
    )

    hardware_tvd = calculate_tvd_from_probs(
        ideal_probs,
        hardware_probs
    )

    real_validation_results.append({
        "validation_index": i,
        "data_row_index": data_idx,
        "reps": int(row["reps"]),
        "entanglement": row["entanglement"],
        "fake_torino_tvd": float(row["mean_tvd"]),
        "fake_kingston_tvd": float(
            fake_kingston_validation_df.iloc[i]["kingston_mean_tvd"]
        ),
        "real_kingston_tvd": float(hardware_tvd),
    })

real_validation_df = pd.DataFrame(
    real_validation_results
)

print(real_validation_df)

    validation_index  data_row_index  reps entanglement  fake_torino_tvd  fake_kingston_tvd  real_kingston_tvd
0                  0              65     1         full         0.065820           0.076953           0.099609
1                  1              44     1       linear         0.070313           0.066797           0.083984
2                  2              21     1       linear         0.069336           0.063867           0.074219
3                  3              17     1       linear         0.072852           0.067578           0.083008
4                  4              67     2         full         0.195166           0.127191           0.167147
5                  5              46     3       linear         0.164342           0.099062           0.110502
6                  6              21     3       linear         0.331858           0.110679           0.089919
7                  7              17     3       linear         0.298367           0.119019           0.101927
8

In [90]:
print(
    "FakeKingston vs Real Kingston:"
)

print(
    real_validation_df[
        ["fake_kingston_tvd", "real_kingston_tvd"]
    ].corr().iloc[0, 1]
)

print(
    "\nFakeTorino vs Real Kingston:"
)

print(
    real_validation_df[
        ["fake_torino_tvd", "real_kingston_tvd"]
    ].corr().iloc[0, 1]
)

FakeKingston vs Real Kingston:
0.8259812524496929

FakeTorino vs Real Kingston:
0.4146539242075321


In [91]:
fake_kingston_mae = np.mean(
    np.abs(
        real_validation_df["fake_kingston_tvd"]
        -
        real_validation_df["real_kingston_tvd"]
    )
)

fake_torino_mae = np.mean(
    np.abs(
        real_validation_df["fake_torino_tvd"]
        -
        real_validation_df["real_kingston_tvd"]
    )
)

print(
    f"FakeKingston → real Kingston MAE: "
    f"{fake_kingston_mae:.4f}"
)

print(
    f"FakeTorino → real Kingston MAE: "
    f"{fake_torino_mae:.4f}"
)

FakeKingston → real Kingston MAE: 0.0229
FakeTorino → real Kingston MAE: 0.1294


## Sanity checks before training a predictor
(PENDING)
Worth confirming before moving on:
- Does `mean_tvd` correlate with `depth` / `cx_count` the way we'd
  expect (more depth/entangling gates → more error)?
- Do `mean_cz_error` / `mean_t1` / `mean_t2` actually vary across the
  dataset (confirming the fully-crossed layout design worked), rather
  than being constant or confounded with row identity?

In [ ]:
print("Correlation of mean_tvd with key structural features:")
print(predictor_df[[
    "mean_tvd", "abstract_depth", "abstract_cx_count",
    "transpiled_depth", "transpiled_cz_count",
    "mean_cz_error", "mean_t1", "mean_t2",
]].corr()["mean_tvd"].sort_values(ascending=False))

print("\nVariation check -- hardware calibration features should NOT be constant:")
print(predictor_df[["mean_cz_error", "mean_t1", "mean_t2"]].describe())

print("\nConfound check -- layout should vary independently of data_row_index:")
print(pd.crosstab(predictor_df["data_row_index"] % 5, predictor_df["layout_id"]).head())


## Three-tier feature ablation

The central scientific question of this study: **does hardware-aware
information improve prediction of noise beyond ordinary circuit
complexity?**

Rather than training one model on the full feature table, we define
three nested feature tiers and compare predictive performance across
them.

In [ ]:
TIER_1_BASELINE = [
    "abstract_n_qubits",
    "abstract_depth",
    "abstract_two_qubit_gate_count",
    "reps",           # will need categorical/numeric encoding for entanglement
]
# entanglement is categorical -- handled via one-hot encoding below

TIER_2_STRUCTURAL = TIER_1_BASELINE + [
    "abstract_total_gate_count",
    "abstract_single_qubit_gate_count",
    "abstract_two_qubit_gate_ratio",
    "abstract_average_gates_per_qubit",
    "abstract_max_gates_per_qubit",
    "abstract_unique_2q_pairs",
    "abstract_average_qubit_degree",
    "abstract_max_qubit_degree",
    "abstract_repeated_2q_pairs",
    "transpiled_depth",
    "transpiled_total_gates",
    "transpiled_1q_gates",
    "transpiled_2q_gates",
    "transpiled_cz_count",
    "transpiled_sx_count",
    "transpiled_rz_count",
]

TIER_3_HARDWARE_AWARE = TIER_2_STRUCTURAL + [
    "mean_cz_error",
    "std_cz_error",
    "min_cz_error",
    "max_cz_error",
    "sum_cz_error",
    "cumulative_cz_error",
    "mean_cz_duration",
    "max_cz_duration",
    "total_cz_duration",
    "physical_qubit_count",
    "mean_t1",
    "min_t1",
    "max_t1",
    "mean_t2",
    "min_t2",
    "max_t2",
]

print("Tier 1 (baseline):", len(TIER_1_BASELINE), "features")
print("Tier 2 (structural):", len(TIER_2_STRUCTURAL), "features")
print("Tier 3 (hardware-aware):", len(TIER_3_HARDWARE_AWARE), "features")


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd

def prepare_features(df, feature_columns):
    X = df[feature_columns].copy()
    # one-hot encode entanglement if present in this tier
    if "entanglement" in df.columns and "entanglement" not in X.columns:
        pass  # entanglement not in this tier's numeric list
    X = pd.get_dummies(X, columns=[c for c in ["entanglement"] if c in X.columns])
    return X.fillna(0)  # T1/T2 NaNs (if any) filled conservatively; revisit if this matters

def evaluate_tier(df, feature_columns, target="mean_tvd", n_folds=5):
    X = prepare_features(df, feature_columns)
    y = df[target]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    cv = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    scores_r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
    scores_neg_mae = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error")

    return {
        "n_features": X.shape[1],
        "mean_r2": scores_r2.mean(),
        "std_r2": scores_r2.std(),
        "mean_mae": -scores_neg_mae.mean(),
    }

# Include entanglement explicitly for tiers that should have it
tier_results = {
    "Tier 1: Baseline": evaluate_tier(predictor_df, TIER_1_BASELINE + ["entanglement"]),
    "Tier 2: Structural": evaluate_tier(predictor_df, TIER_2_STRUCTURAL + ["entanglement"]),
    "Tier 3: Hardware-aware": evaluate_tier(predictor_df, TIER_3_HARDWARE_AWARE + ["entanglement"]),
}

tier_comparison = pd.DataFrame(tier_results).T
print(tier_comparison)


**Interpreting this table:** if `mean_r2` increases meaningfully from
Tier 1 → Tier 2 → Tier 3, that's evidence hardware-aware calibration
data genuinely improves noise prediction beyond circuit structure
alone -- the paper's central claim. If Tier 3 doesn't improve over
Tier 2, that's still a valid, honest, reportable finding (structure
alone may be sufficient, or the hardware features may need
refinement/more data to show their value).

For feature importance *within* a tier (e.g. which hardware features
matter most in Tier 3), prefer **permutation importance** over default
impurity-based importance, since several hardware features (CZ count,
sum/mean/cumulative CZ error) are correlated and impurity-based
importance can split credit misleadingly across them.

In [ ]:
from sklearn.inspection import permutation_importance

X3 = prepare_features(predictor_df, TIER_3_HARDWARE_AWARE + ["entanglement"])
y3 = predictor_df["mean_tvd"]

model3 = RandomForestRegressor(n_estimators=200, random_state=42)
model3.fit(X3, y3)

perm_importance = permutation_importance(
    model3, X3, y3, n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X3.columns,
    "importance_mean": perm_importance.importances_mean,
    "importance_std": perm_importance.importances_std,
}).sort_values("importance_mean", ascending=False)

print(importance_df.head(15).to_string(index=False))
